In [5]:
import pandas as pd
import re
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('messy_sales_data.csv')

# 1. Drop unnecessary columns
df.drop(columns=['Notes', 'Salesperson'], inplace=True, errors='ignore')

# 2. Clean categorical columns
categorical_cols = ['Order ID', 'Customer Name', 'Product', 'Payment Method', 'Status', 'Category', 'City']

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown').astype(str).str.strip()
    
    if col == 'City':
        # lowercase, remove special chars, empty replace
        df['City'] = (
            df['City']
            .str.lower()
            .str.replace(r'[^a-z\s]', '', regex=True)
            .replace('', 'Unknown')
        )

# 3. Clean numeric columns
numeric_cols = ['Quantity', 'Unit Price', 'Total Amount']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# 4. Clean order date
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Order Date'] = df['Order Date'].fillna(df['Order Date'].mode()[0])
df['Order Date'] = df['Order Date'].dt.strftime('%Y-%m-%d')

# 5. Remove duplicates
# print(df[df.duplicated()])
# print(df[df['Order ID'].isin(['ORD-1510-17'])])
df.drop_duplicates(subset='Order ID', inplace=True)

# 6. Check outliers
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    
correlation_matrix = df[numeric_cols].corr()

print(correlation_matrix)

# 7. Save cleaned dataset
# df.to_csv('messy_sales_data_cleaned.csv', index=False)
# print("\n Cleaned dataset saved as 'messy_sales_data_cleaned.csv'")


C:\Users\Omar Faruk\AppData\Local\Temp\ipykernel_9356\1972929783.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')


              Quantity  Unit Price  Total Amount
Quantity      1.000000    0.043609      0.866081
Unit Price    0.043609    1.000000      0.107022
Total Amount  0.866081    0.107022      1.000000
